# NYC Fire Hydrant Density Analysis

This notebook analyzes the spatial distribution of fire hydrants across New York City neighborhoods.

**Workflow:**
1. Load raw hydrant and neighborhood boundary datasets
2. Project both datasets to a common, distance-accurate CRS
3. Spatially join hydrants to their containing neighborhood
4. Compute hydrant density (hydrants per km²) for each neighborhood
5. Visualize density on an interactive choropleth map

In [1]:
import os
import geopandas as gpd

# GeoPandas 1.0+ is required for the GeoDataFrame.explore() choropleth used later in this notebook
print("GeoPandas version:", gpd.__version__)

GeoPandas version: 1.0.1


## 1. Load Data

Read the raw hydrant and neighborhood GeoJSON files (sourced from NYC Open Data) and keep only the
columns needed for this analysis. Both datasets are published in EPSG:4326 (WGS84 lat/lon).

In [2]:
# Read the source GeoJSON files
hydrants = gpd.read_file('./data/raw/hydrants.geojson')
hydrants = hydrants[['boro', 'latitude', 'longitude', 'geometry']]

neighborhoods = gpd.read_file('./data/raw/neighborhoods.geojson')
# ntaname/nta2020 identify each Neighborhood Tabulation Area; countyfips is kept for reference
neighborhoods = neighborhoods[['boroname', 'borocode', 'ntaname', 'nta2020', 'countyfips', 'geometry']]

print('*' * 60)
print('Hydrants shape', hydrants.shape, 'CRS', hydrants.crs)
print('Neighborhoods shape', neighborhoods.shape, 'CRS', neighborhoods.crs)

************************************************************
Hydrants shape (109725, 4) CRS EPSG:4326
Neighborhoods shape (262, 6) CRS EPSG:4326


## 2. Project to a Common CRS

WGS84 (EPSG:4326) is a geographic CRS measured in degrees, so it can't be used to measure distance
or area accurately. Reproject both GeoDataFrames to EPSG:2263 (NY State Plane, Long Island, feet),
a projected CRS suitable for accurate spatial joins and area calculations within NYC.

In [3]:
# Inspect unique neighborhood names before projection
print('Sample neighborhood values:', neighborhoods['ntaname'].unique()[:10])

hydrants = hydrants.to_crs(epsg=2263)
neighborhoods = neighborhoods.to_crs(epsg=2263)

print('Hydrants shape', hydrants.shape, 'CRS', hydrants.crs)
print('Neighborhoods shape', neighborhoods.shape, 'CRS', neighborhoods.crs)

Sample neighborhood values: ['Greenpoint' 'Williamsburg' 'South Williamsburg' 'East Williamsburg'
 'Brooklyn Heights' 'Downtown Brooklyn-DUMBO-Boerum Hill' 'Fort Greene'
 'Clinton Hill' 'Brooklyn Navy Yard' 'Bedford-Stuyvesant (West)']
Hydrants shape (109725, 4) CRS EPSG:2263
Neighborhoods shape (262, 6) CRS EPSG:2263


## 3. Spatial Join: Assign Neighborhoods to Hydrants

Use a "within" spatial join to tag each hydrant with the neighborhood polygon that contains it,
then aggregate to get a hydrant count per neighborhood. A small number of hydrants may not fall
within any mapped neighborhood polygon (e.g. hydrants on piers, causeways, or other unmapped land) —
these are captured separately as `unmatched` rather than silently dropped.

In [4]:
# Spatially join hydrants to the neighborhood polygons
hydrants_with_neighborhood = gpd.sjoin(hydrants, neighborhoods, how='left', predicate='within')

print(f"Hydrants with neighborhood info: {hydrants_with_neighborhood.shape}")

# Group by neighborhood and count hydrants per neighborhood
hydrant_counts = hydrants_with_neighborhood.groupby('ntaname').size().reset_index(name='hydrant_count')

# Hydrants which didn't fall inside any polygon (index_right is NaN when the left join found no match)
unmatched = hydrants_with_neighborhood[hydrants_with_neighborhood['index_right'].isna()]
print("Hydrants which didn't fall inside any polygon:", len(unmatched))

unmatched[['latitude', 'longitude']].head(10)

hydrant_counts

Hydrants with neighborhood info: (109725, 10)
Hydrants which didn't fall inside any polygon: 31


,ntaname,hydrant_count
0,Allerton,280
1,Alley Pond Park,48
2,Annadale-Huguenot-Prince's Bay-Woodrow,1708
3,Arden Heights-Rossville,717
4,Astoria (Central),348
...,...,...
254,Windsor Terrace-South Slope,305
255,Woodhaven,516
256,Woodlawn Cemetery,13
257,Woodside,566


## 4. Hydrant Density per Neighborhood

Compute each neighborhood's area in km² (converting from the EPSG:2263 unit, square feet), merge in
the hydrant counts from the spatial join, and divide to get hydrant density. Neighborhoods with no
hydrant match get a count of 0. Neighborhoods with an (effectively) zero mapped land area — e.g. tiny
islands or park slivers — would otherwise divide-by-zero into `inf`/`NaN`; these are guarded and
marked as missing data instead so they don't distort sorting or the density map below. The full
(unrounded) density value is kept for classification and only rounded when displayed, so the
choropleth map's quantile bins are computed from real precision, not display-rounded values.

In [5]:
import numpy as np

SQ_FT_PER_KM2 = 10_763_910.42

neighborhoods['Area_km2'] = neighborhoods.geometry.area / SQ_FT_PER_KM2

result = neighborhoods.merge(hydrant_counts, on='ntaname', how='left').fillna({'hydrant_count': 0})

# Guard against divide-by-zero for neighborhoods with (near) zero mapped land area: mark as NaN
# ("no data") instead of letting them evaluate to inf and distort sorting/classification below.
density = np.where(
    result['Area_km2'] > 0,
    result['hydrant_count'] / result['Area_km2'],
    np.nan,
)
result['Density_perSqkm'] = density

# Round only for display here; `result['Density_perSqkm']` itself stays full-precision for the
# quantile classification used by the choropleth map in the next section.
result.sort_values('Density_perSqkm', ascending=False).assign(
    Density_perSqkm=lambda d: d['Density_perSqkm'].round(2)
)

,boroname,borocode,ntaname,nta2020,countyfips,geometry,Area_km2,hydrant_count,Density_perSqkm
132,Manhattan,1,Gramercy,MN0602,061,"MULTIPOLYGON (((990196.892 207745.371, 990187....",0.699188,269.0,384.73
121,Manhattan,1,SoHo-Little Italy-Hudson Square,MN0201,061,"MULTIPOLYGON (((983469.159 204638.902, 983496....",1.200006,432.0,360.00
119,Manhattan,1,Tribeca-Civic Center,MN0102,061,"MULTIPOLYGON (((984440.604 200699.422, 984402....",1.261461,433.0,343.25
123,Manhattan,1,West Village,MN0203,061,"MULTIPOLYGON (((981713.541 209788.14, 981751 2...",1.339370,447.0,333.74
118,Manhattan,1,Financial District-Battery Park City,MN0101,061,"MULTIPOLYGON (((984032.884 192223.749, 983984....",1.786223,570.0,319.11
...,...,...,...,...,...,...,...,...,...
68,Brooklyn,3,Shirley Chisholm State Park,BK5693,047,"MULTIPOLYGON (((1019419.268 173893.122, 101946...",1.698704,2.0,1.18
258,Staten Island,5,Fort Wadsworth,SI9561,085,"MULTIPOLYGON (((967656.829 155637.132, 967549....",0.916692,1.0,1.09
13,Brooklyn,3,The Evergreens Cemetery,BK0471,047,"MULTIPOLYGON (((1012948.8 187911.295, 1012925....",0.518903,0.0,0.00
67,Brooklyn,3,Jamaica Bay (West),BK5692,047,"MULTIPOLYGON (((1022227.32 152028.146, 1022078...",3.693179,0.0,0.00


## 5. Visualize: Hydrant Density Choropleth Map

Render an interactive choropleth of `Density_perSqkm` per neighborhood using GeoDataFrame.explore()
(GeoPandas' Folium/Leaflet wrapper). Neighborhoods with no data (zero mapped land area, see above)
are styled distinctly rather than mixed into the color scale.

In [ ]:
# folium/leaflet expects geographic coordinates, so reproject back to EPSG:4326 for display
result_wgs84 = result.to_crs(epsg=4326)

# Round only for the tooltip; the underlying 'Density_perSqkm' column stays full precision
# so the quantile bin edges below are computed accurately.
result_wgs84['Density_perSqkm_display'] = result_wgs84['Density_perSqkm'].round(2)

m = result_wgs84.explore(
    column='Density_perSqkm',
    cmap='YlOrRd',
    scheme='quantiles',
    k=6,
    legend=True,
    tooltip=['ntaname', 'boroname', 'hydrant_count', 'Density_perSqkm_display'],
    style_kwds={'color': 'black', 'weight': 0.3},
    # Grey out neighborhoods with no computable density (zero mapped land area) instead of
    # letting NaN break or silently disappear from the quantile classification
    missing_kwds={'color': 'lightgrey', 'label': 'No data'},
    legend_kwds={'caption': 'Hydrant Density (per sq km)'},
    tiles='CartoDB positron',
    name='Hydrant Density'
)

m